# Olist Data Cleaning

This notebook loads the raw Olist files, standardizes fields, removes duplicates, handles missing values, and saves clean CSV files in `Clean_Data`. 

In [ ]:
from pathlib import Path
import pandas as pd

base = Path.cwd()
if not (base / 'Raw_Data').exists():
    base = Path('../..').resolve()
raw_path = base / 'Raw_Data'
clean_path = base / 'Clean_Data'
clean_path.mkdir(exist_ok=True)
files = {'customers':'olist_customers_dataset.csv','geolocation':'olist_geolocation_dataset.csv','orders':'olist_orders_dataset.csv','order_items':'olist_order_items_dataset.csv','order_payments':'olist_order_payments_dataset.csv','order_reviews':'olist_order_reviews_dataset.csv','products':'olist_products_dataset.csv','sellers':'olist_sellers_dataset.csv','category_translation':'product_category_name_translation.csv'}
data = {name: pd.read_csv(raw_path / file) for name, file in files.items()}
{name: frame.shape for name, frame in data.items()}

## Standardize text and dates

In [ ]:
def clean_text(frame, columns):
    for column in columns:
        if column in frame:
            frame[column] = frame[column].astype('string').str.strip().str.lower()
    return frame

text_columns = {'customers':['customer_city','customer_state'],'geolocation':['geolocation_city','geolocation_state'],'orders':['order_status'],'order_payments':['payment_type'],'order_reviews':['review_comment_title','review_comment_message'],'products':['product_category_name'],'sellers':['seller_city','seller_state'],'category_translation':['product_category_name','product_category_name_english']}
for name, columns in text_columns.items():
    data[name] = clean_text(data[name], columns)
date_columns = {'orders':['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'],'order_items':['shipping_limit_date'],'order_reviews':['review_creation_date','review_answer_timestamp']}
for name, columns in date_columns.items():
    for column in columns:
        data[name][column] = pd.to_datetime(data[name][column], errors='coerce')

## Remove duplicates and handle missing values

In [ ]:
data['customers'] = data['customers'].drop_duplicates('customer_id').fillna({'customer_city':'unknown','customer_state':'unknown'})
data['orders'] = data['orders'].drop_duplicates('order_id')
data['order_items'] = data['order_items'].dropna(subset=['order_id','product_id','seller_id']).drop_duplicates(['order_id','order_item_id'])
data['order_payments'] = data['order_payments'].dropna(subset=['order_id']).drop_duplicates(['order_id','payment_sequential'])
data['order_reviews'] = data['order_reviews'].drop_duplicates('review_id').fillna({'review_comment_title':'','review_comment_message':''})
data['products'] = data['products'].drop_duplicates('product_id')
data['products']['product_category_name'] = data['products']['product_category_name'].fillna('unknown')
data['sellers'] = data['sellers'].drop_duplicates('seller_id').fillna({'seller_city':'unknown','seller_state':'unknown'})
data['category_translation'] = data['category_translation'].drop_duplicates('product_category_name').fillna('unknown')
numeric_columns = {'order_items':['order_item_id','price','freight_value'],'order_payments':['payment_sequential','payment_installments','payment_value'],'order_reviews':['review_score'],'products':['product_name_lenght','product_description_lenght','product_photos_qty','product_weight_g','product_length_cm','product_height_cm','product_width_cm']}
for name, columns in numeric_columns.items():
    for column in columns:
        data[name][column] = pd.to_numeric(data[name][column], errors='coerce')
for column in ['payment_installments','payment_value']:
    data['order_payments'][column] = data['order_payments'][column].fillna(0)
data['order_reviews']['review_score'] = data['order_reviews']['review_score'].fillna(0)
for column in ['product_name_lenght','product_description_lenght','product_photos_qty']:
    data['products'][column] = data['products'][column].fillna(0)

## Reduce geolocation to one row per ZIP prefix

In [ ]:
geo = data['geolocation'].copy()
geo['geolocation_zip_code_prefix'] = pd.to_numeric(geo['geolocation_zip_code_prefix'], errors='coerce')
geo = geo.dropna(subset=['geolocation_zip_code_prefix'])
geo['geolocation_city'] = geo['geolocation_city'].fillna('unknown')
geo['geolocation_state'] = geo['geolocation_state'].fillna('unknown')
data['geolocation'] = geo.groupby('geolocation_zip_code_prefix', as_index=False).agg({'geolocation_lat':'median','geolocation_lng':'median','geolocation_city':'first','geolocation_state':'first'})

## Save the clean files

In [ ]:
for name, frame in data.items():
    frame.to_csv(clean_path / f'{name}_clean.csv', index=False)
summary = pd.DataFrame({'file':[f'{name}_clean.csv' for name in data],'rows':[frame.shape[0] for frame in data.values()],'columns':[frame.shape[1] for frame in data.values()]})
summary